In [1]:
import numpy as np
import pandas as pd
import random as rd
import seaborn as sns
from scipy import interp
import scipy.stats as st
import os
import copy
import re
import time
from copy import deepcopy
import matplotlib.pyplot as plt
import cloudpickle as pickle
from scipy import stats
from sklearn.metrics import roc_curve, auc,accuracy_score, precision_score, recall_score,f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Lasso
from sklearn import *
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

In [2]:
MLA = [
    #Ensemble Methods
    ensemble.GradientBoostingClassifier(),
    ensemble.RandomForestClassifier(),
    #GLM
    linear_model.LogisticRegression(),
    #NN
    neural_network.MLPClassifier()
]

# def function 

## Intra-population cross-validation

In [3]:

def get_self_cv(alg, data, Groups, k=5, SEED = 5):
    splitor = StratifiedKFold(n_splits=k, shuffle=True,random_state=SEED)
    ### alg choice
    if alg == 'gbc':
        clf = GradientBoostingClassifier(n_estimators=101, learning_rate=0.1, subsample=1,loss='deviance',
                                         max_features=0.1, min_samples_leaf=1, random_state=SEED,
                                         criterion='friedman_mse',max_depth=3)
    if alg == 'rfc':
        clf = RandomForestClassifier(n_estimators = 501, oob_score = True, random_state =SEED,class_weight='balanced', 
                                max_features = 'log2', criterion='gini', n_jobs = 2) ###max_depth=5,min_samples_leaf=10,class_weight='balanced'
    if alg == 'lrcv':
        clf = LogisticRegression(penalty='l2', dual=False, C=1.0, fit_intercept=True, intercept_scaling=1,
                                 class_weight=None, random_state=SEED, solver='liblinear', 
                                 max_iter=100, multi_class='ovr', verbose=0, warm_start=False, n_jobs=1)
    if alg == 'mlpc':
        clf = MLPClassifier(solver='lbfgs', alpha=1e-5,hidden_layer_sizes=(10,5), random_state=SEED) 

    aucs = []
    accs = []
    percisions = []
    recalls = []
    F1s = []
    i = 0
    for train_index, test_index in splitor.split(data, Groups):
        y_train, y_test = Groups[train_index], Groups[test_index]
        X_train, X_test = np.array(data)[train_index], np.array(data)[test_index]
        pred = clf.fit(X_train, y_train).predict(X_test)
        probas = clf.fit(X_train, y_train).predict_proba(X_test)
        fpr, tpr, thresholds = roc_curve(y_test, probas[:, 1])
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        acc = accuracy_score(y_test, pred)
        accs.append(acc)
        percision = precision_score(y_test, pred)
        percisions.append(percision)
        recall = recall_score(y_test, pred)
        recalls.append(recall)
        f1score = f1_score(y_test,pred)
        F1s.append(f1score)
        i += 1
    Auc = np.mean(aucs)
    Recall = np.mean(recalls)
    Precision = np.mean(percisions)
    F1_score = np.mean(F1s)
    Accuracy = np.mean(accs)
    #Auc = max(aucs)
    #Recall = max(recalls)
    #Precision = max(percisions)
    #F1_score = max(F1s)
    #Accuracy = max(accs)
    return Auc, Recall, Precision, F1_score, Accuracy,clf 
      

In [4]:
def get_lodo_auc(alg, data,site, SEED = 5):
    
    ### alg choice
    if alg == 'gbc':
        clf = GradientBoostingClassifier(n_estimators=101, learning_rate=0.1, subsample=1,loss='deviance',
                                         max_features=0.1, min_samples_leaf=1, random_state=SEED,
                                         criterion='friedman_mse',max_depth=3)
    if alg == 'rfc':
        clf = RandomForestClassifier(n_estimators = 501, oob_score = True, random_state =SEED, class_weight='balanced',
                                max_features = 'log2',criterion='gini', n_jobs = 2)#, max_depth=5
    if alg == 'lrcv':
        clf = LogisticRegression(penalty='l2', dual=False, C=1.0, fit_intercept=True, intercept_scaling=1,
                                 class_weight=None, random_state=SEED, solver='liblinear', 
                                 max_iter=100, multi_class='ovr', verbose=0, warm_start=False, n_jobs=1)
    if alg == 'mlpc':
        clf = MLPClassifier(solver='lbfgs', alpha=1e-5,hidden_layer_sizes=(10,5), random_state=SEED)

    ###data
    X_train = data.loc[data['Study'] != site,Feature]
    X_test = data.loc[data['Study'] == site,Feature]
    y_train_label = data.loc[data['Study'] != site,'Group']
    y_train = np.array([0 if i == 'CTR' else 1 for i in y_train_label])
    y_test_label = data.loc[data['Study'] == site,'Group']
    y_test = np.array([0 if i == 'CTR' else 1 for i in y_test_label])        
    pred = clf.fit(X_train, y_train).predict(X_test)
    probas = clf.fit(X_train, y_train).predict_proba(X_test)
    fpr, tpr, thresholds = roc_curve(y_test, probas[:, 1])
    Auc = auc(fpr, tpr)
    Accuracy = accuracy_score(y_test, pred)
    Percision = precision_score(y_test, pred)
    Recall = recall_score(y_test, pred)
    F1_score = f1_score(y_test,pred)
    return Auc, Recall, Precision, F1_score, Accuracy 

## Population-to-population analysis 

In [6]:
def get_studytostudy_auc(alg, data,site_train, site_test,SEED = 5):
    
    ### alg choice
    if alg == 'gbc':
        clf = GradientBoostingClassifier(n_estimators=101, learning_rate=0.1, subsample=1,loss='deviance',
                                         max_features=0.1, min_samples_leaf=1, random_state=SEED,
                                         criterion='friedman_mse',max_depth=3)
    if alg == 'rfc':
        clf = RandomForestClassifier(n_estimators = 501, oob_score = True, random_state =SEED, class_weight='balanced',
                                max_features = 'log2',criterion='gini', n_jobs = 2)##, max_depth=5
    if alg == 'lrcv':
        clf = LogisticRegression(penalty='l2', dual=False, C=1.0, fit_intercept=True, intercept_scaling=1,
                                 class_weight=None, random_state=SEED, solver='liblinear', 
                                 max_iter=100, multi_class='ovr', verbose=0, warm_start=False, n_jobs=1)
    if alg == 'mlpc':
        clf = MLPClassifier(solver='lbfgs', alpha=1e-5,hidden_layer_sizes=(10,5), random_state=SEED)
    
    X_train = data.loc[data['Study'] == site_train,Feature]
    X_test = data.loc[data['Study'] == site_test,Feature]
    y_train_label = data.loc[data['Study'] == site_train,'Group']
    y_train = np.array([0 if i == 'CTR' else 1 for i in y_train_label])
    y_test_label = data.loc[data['Study'] == site_test,'Group']
    y_test = np.array([0 if i == 'CTR' else 1 for i in y_test_label])
    pred = clf.fit(X_train, y_train).predict(X_test)
    probas = clf.fit(X_train, y_train).predict_proba(X_test)
    fpr, tpr, thresholds = roc_curve(y_test, probas[:, 1])
    Auc = auc(fpr, tpr)
    Accuracy = accuracy_score(y_test, pred)
    Percision = precision_score(y_test, pred)
    Recall = recall_score(y_test, pred)
    F1_score = f1_score(y_test,pred)
    return Auc, Recall, Precision, F1_score, Accuracy 

## LOPO_analysis

In [5]:
def get_lodo_val(alg, data_train,data,site, SEED = 5):
    
    ### alg choice
    if alg == 'gbc':
        clf = GradientBoostingClassifier(n_estimators=101, learning_rate=0.1, subsample=1,loss='deviance',
                                         max_features=0.1, min_samples_leaf=1, random_state=SEED,
                                         criterion='friedman_mse',max_depth=3)
    if alg == 'rfc':
        clf = RandomForestClassifier(n_estimators = 501, oob_score = True, random_state =SEED, class_weight='balanced',
                                max_features = 'log2',criterion='gini', n_jobs = 2)##, max_depth=5
    if alg == 'lrcv':
        clf = LogisticRegression(penalty='l2', dual=False, C=1.0, fit_intercept=True, intercept_scaling=1,
                                 class_weight=None, random_state=SEED, solver='liblinear', 
                                 max_iter=100, multi_class='ovr', verbose=0, warm_start=False, n_jobs=1)
    if alg == 'mlpc':
        clf = MLPClassifier(solver='lbfgs', alpha=1e-5,hidden_layer_sizes=(10,5), random_state=SEED)

    ###data
    X_train = data_train.loc[:,Feature]
    X_test = data.loc[data['Study'] == site,Feature]
    y_train_label = data_train.loc[:,'Group']
    y_train = np.array([0 if i == 'CTR' else 1 for i in y_train_label])
    y_test_label = data.loc[data['Study'] == site,'Group']
    y_test = np.array([0 if i == 'CTR' else 1 for i in y_test_label])        
    pred = clf.fit(X_train, y_train).predict(X_test)
    probas = clf.fit(X_train, y_train).predict_proba(X_test)
    fpr, tpr, thresholds = roc_curve(y_test, probas[:, 1])
    Auc = auc(fpr, tpr)
    Accuracy = accuracy_score(y_test, pred)
    Percision = precision_score(y_test, pred)
    Recall = recall_score(y_test, pred)
    F1_score = f1_score(y_test,pred)
    return Auc, Recall, Precision, F1_score, Accuracy 

# Data_prepare

In [9]:
data_feat=pd.read_csv("/Users/jiaona/CRC_fungi/NewProfile/CRC_fungi_merged_species_rela.csv",index_col=0)
data_feat_all=data_feat.T[CommonID]
data_feat_all.shape

(1545, 28)

In [10]:
metadata=pd.read_csv("/Users/jiaona/CRC_fungi/metadata/CRC_count_metadata_CHI_JAP.csv",index_col=1)
meta_all=metadata.loc[(metadata['Group']!='adenoma')& (metadata['Group']!='HS'),['Run','Study','Group','BMI','Age']]
meta_cn=metadata.loc[(metadata['Dataset']=='Discovery')& (metadata['Group']!='adenoma')& 
                     (metadata['Group']!='HS'),['Run','Study','Group','BMI','Age']]
meta_cn.head()
meta_cn.shape

(985, 5)

In [11]:
data_cn = pd.merge(meta_cn,data_feat_all,left_on='Run',right_index=True)
print(data_cn.shape)
data_cn.head()

(985, 33)


,Run,Study,Group,BMI,Age,Macrophomina phaseolina,Parastagonospora nodorum,Fonsecaea erecta,Aspergillus clavatus,Aspergillus niger,...,Nadsonia fulvescens,Fusarium graminearum,Fusarium proliferatum,Hypsizygus marmoreus,Moniliophthora perniciosa,Sistotremastrum niveocremeum,Sistotremastrum suecicum,Nosema bombycis,Mucor ambiguus,Rhizopus microsporus
projectID,,,,,,,,,,,,,,,,,,,,,
DRA006684,DRR127476,JAP,CRC,22.460034,64.0,0.002627,0.001082,0.000920,0.000963,0.006507,...,0.000635,0.001035,0.000253,0.002950,0.000925,0.000197,0.000576,0.003821,0.002354,0.056627
DRA006684,DRR127478,JAP,CRC,22.600263,66.0,0.003795,0.001206,0.000976,0.000464,0.006690,...,0.000323,0.000613,0.000285,0.004280,0.000528,0.000040,0.000270,0.002728,0.003288,0.085655
DRA006684,DRR127481,JAP,CRC,28.293345,61.0,0.001453,0.001069,0.000649,0.001069,0.003344,...,0.000910,0.000475,0.000455,0.005559,0.000096,0.000190,0.000247,0.000924,0.002217,0.117581
DRA006684,DRR127485,JAP,CRC,21.907583,70.0,0.002987,0.000919,0.000699,0.000606,0.007286,...,0.000568,0.000748,0.000260,0.005592,0.000442,0.000320,0.000282,0.000640,0.003397,0.111794
DRA006684,DRR127488,JAP,CTR,22.136740,72.0,0.001761,0.000602,0.000319,0.000766,0.009295,...,0.000641,0.000462,0.000475,0.006037,0.000567,0.000482,0.000472,0.000722,0.001800,0.123720


In [12]:
site=list(set(data_cn['Study']))
site.sort()
print(site)

['AUS', 'CHI', 'FRA', 'GER', 'JAP']


# Classification models

In [17]:
Feature=[]
fileid=open("temp/CRC_fungi_feature_select.txt",'r')
for line in fileid:
    Feature.append(line.strip().split('\t')[0])
print(len(Feature))

24


In [18]:
#site=['AUS', 'CHI', 'FRA', 'GER', 'JAP', 'USA', 'ITA1']
rd.seed(123)
time_start=time.time()
random_times = 10 
MLA = ['gbc','rfc','lrcv','mlpc']
Site_Result = {}

for s in site:
    print(s)
    random_columns = ['AUC','Recall','Precision','F1','accuracy']
    random_results = pd.DataFrame(columns = random_columns)
    data = data_cn.loc[data_cn['Study'] == s,]
    data_feat = data.loc[:,Feature]
    group = data['Group']
    print(data_feat.shape)
    print(len(group))
    label = group.replace({'CTR':0,'CRC':1})
    label.columns = ['Group']
    label = label.to_frame()
    label = np.array([i for i in label.Group])
    for i in range(random_times):
        Auc, Recall, Precision, F1_score, Accuracy, clf  = get_self_cv(alg='rfc', data=data_feat,Groups=label, k=10, SEED = 2*i) 
        #print(Auc, Recall, Precision, F1_score, Accuracy)
        random_results.loc[i,:] = [Auc, Recall, Precision, F1_score, Accuracy]
    Result_mean = random_results.max()
    #Result_mean = random_results.mean()
    print(Result_mean[0])
    Site_Result[s] = Result_mean
    
Site_Result = pd.DataFrame(Site_Result)
print(np.mean(Site_Result.iloc[0,:]))
Site_Result
#Site_Result.to_csv('/Users/jiaona/CRC_fungi/temp/CRC_fungi_gbc_selfcv.csv')
time_end=time.time()
print('time cost',time_end-time_start,'s')



AUS
(109, 24)
109
0.8426190476190477
CHI
(128, 24)
128
0.742797619047619
FRA
(114, 24)
114
0.8830952380952383
GER
(125, 24)
125
0.8486111111111111
JAP
(509, 24)
509
0.6488273372781064
0.7931900706302244
time cost 1120.0511012077332 s


In [217]:
print(np.mean(Site_Result.iloc[0,:]))

0.7857075501080116


In [193]:
print(np.mean(Site_Result.iloc[0,:]))

0.7545625101906641


In [229]:
Site_Result

,AUS,CHI,FRA,GER,JAP
AUC,0.835833,0.724583,0.899365,0.824405,0.644351
Recall,0.650000,0.814286,0.676667,0.733333,0.642923
Precision,0.801667,0.681732,0.816667,0.744762,0.591843
F1,0.687929,0.736447,0.729697,0.728958,0.613562
accuracy,0.762121,0.669048,0.770221,0.749359,0.595186


In [27]:
Site_Result.to_csv('temp/CRC_fungi_rfc_selfcvP.csv')

## LOPO_test 

In [19]:
#Feature = CommonID+['Age']
Feature = Feature
#Feature = CommonID
print(len(Feature))
LODO_Result = {}
for s in site:
    print(s)
    Auc, Recall, Precision, F1_score, Accuracy  =  get_lodo_auc(alg='rfc', data=data_cn, site=s,SEED = 123)
    LODO_Result[s] = [Auc, Recall, Precision, F1_score, Accuracy]
LODO_Result = pd.DataFrame(LODO_Result)
LODO_Result.index = ['Auc','Recall','Precision','F1_score','Accuracy']
LODO_Result

24
AUS
CHI
FRA
GER
JAP


,AUS,CHI,FRA,GER,JAP
Auc,0.803830,0.651902,0.744974,0.756667,0.615932
Recall,0.608696,0.743243,0.905660,0.733333,0.104651
Precision,0.590954,0.590954,0.590954,0.590954,0.590954
F1_score,0.666667,0.705128,0.690647,0.682171,0.182432
Accuracy,0.743119,0.640625,0.622807,0.672000,0.524558


In [17]:
print(np.mean(LODO_Result.iloc[0,:]))

0.7146608254578827


In [28]:
LODO_Result.to_csv('temp/CRC_fungi_rfc_lopo.csv')

## Off_diag(population-population) 

In [20]:
Off_diag_auc = {}
for s in site:
    temp = deepcopy(site)
    temp.remove(s)
    for ss in temp:
        print(s,ss)
        Auc, Recall, Precision, F1_score, Accuracy = get_studytostudy_auc(alg='rfc',data=data_cn,site_train=s,site_test=ss,SEED = 123)
        Off_diag_auc[s+'_'+ss] = [Auc, Recall, Precision, F1_score, Accuracy]
OFF_Result = pd.DataFrame(Off_diag_auc)
OFF_Result.index = ['Auc','Recall','Precision','F1_score','Accuracy']
OFF_Result

AUS CHI
AUS FRA
AUS GER
AUS JAP
CHI AUS
CHI FRA
CHI GER
CHI JAP
FRA AUS
FRA CHI
FRA GER
FRA JAP
GER AUS
GER CHI
GER FRA
GER JAP
JAP AUS
JAP CHI
JAP FRA
JAP GER


,AUS_CHI,AUS_FRA,AUS_GER,AUS_JAP,CHI_AUS,CHI_FRA,CHI_GER,CHI_JAP,FRA_AUS,FRA_CHI,FRA_GER,FRA_JAP,GER_AUS,GER_CHI,GER_FRA,GER_JAP,JAP_AUS,JAP_CHI,JAP_FRA,JAP_GER
Auc,0.586086,0.654964,0.790128,0.586955,0.659938,0.778534,0.708333,0.577141,0.743099,0.687563,0.672949,0.535007,0.759489,0.590966,0.640736,0.582669,0.761732,0.612112,0.552119,0.682436
Recall,0.770270,0.867925,0.683333,0.635659,0.717391,0.811321,0.900000,0.306202,0.413043,0.567568,0.533333,0.034884,0.673913,0.783784,0.905660,0.767442,0.891304,0.851351,0.849057,0.900000
Precision,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954,0.590954
F1_score,0.686747,0.643357,0.677686,0.585714,0.594595,0.693548,0.666667,0.406170,0.513514,0.646154,0.551724,0.065693,0.639175,0.686391,0.631579,0.634615,0.661290,0.684783,0.608108,0.635294
Accuracy,0.593750,0.552632,0.688000,0.544204,0.587156,0.666667,0.568000,0.546169,0.669725,0.640625,0.584000,0.497053,0.678899,0.585938,0.508772,0.552063,0.614679,0.546875,0.491228,0.504000


In [157]:
print(np.mean(LODO_Result.iloc[0,:]))

0.7154932440877422


In [29]:
OFF_Result.to_csv('temp/CRC_fungi_rfc_offdiag.csv')